# 🔭 Universe screener — find symbols worth backfilling

A **universe** is just a named list of symbols in `config/universes.yaml`. This
notebook builds one in four steps, all free and keyless:

1. **Fetch** every US-listed stock from the Nasdaq screener API (~1 minute).
2. **Screen** down to liquid common stocks (drop warrants/funds/preferreds, apply
   market-cap and price floors).
3. **Pick** the top names by market cap, capped per sector — so a backtested edge
   is never a hidden bet on one sector.
4. **Persist** — write the universe YAML and register the symbols in the database.

Re-running is always safe: the YAML merges, the database registration UPSERTs.
If the (unofficial) Nasdaq endpoint is ever down, you can simply hand-edit
`config/universes.yaml` instead — it is a plain `name: [symbols]` list.

## 1 · Parameters

In [ ]:
UNIVERSE_NAME = "LiquidLeaders"   # the name strategies will refer to
TOP_N         = 100               # universe size
MIN_CAP       = 2e9               # market-cap floor in $ (2e9 = $2B)
MIN_PRICE     = 5.0               # price floor in $ (avoids illiquid penny names)
SECTOR_CAP    = 0.25              # no sector may exceed 25% of the universe

## 2 · Set up

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  .venv/bin/pip install -e ".[notebook]"
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import universe as uv

## 3 · Fetch all US listings

One request per exchange (NASDAQ / NYSE / AMEX). The result has one row per
symbol with name, last price, market cap, sector and industry.

In [ ]:
listed = uv.fetch_listed_stocks()
print(f"{len(listed):,} US listings fetched; "
      f"sector known for {listed['sector'].notna().mean():.0%} of them")
listed.head(5)

## 4 · Screen down to liquid common stocks

Rules (see `universe.screen_candidates`): plain 1–5 letter symbol, name must not
be a warrant/unit/fund/preferred/SPAC, market cap ≥ the floor, price ≥ the floor.

In [ ]:
screened = uv.screen_candidates(listed, min_market_cap=MIN_CAP, min_price=MIN_PRICE)
print(f"{len(listed):,} listings → {len(screened):,} pass the screen")

import plotly.graph_objects as go
from trading_simple.plots import SERIES
sectors = screened["sector"].fillna("Unknown").value_counts()
fig = go.Figure(go.Bar(x=sectors.index, y=sectors.values, marker={"color": SERIES[0]}))
fig.update_layout(title=f"Screen survivors by sector ({len(screened):,} names)",
                  template="plotly_white", height=350)
fig.show()

## 5 · Pick the universe (sector-capped)

Greedy top-N by market cap, skipping any sector already at
`SECTOR_CAP × TOP_N` names.

In [ ]:
picks = uv.liquid_leaders(screened, n=TOP_N, sector_cap=SECTOR_CAP)
frame = screened[screened["symbol"].isin(picks)]
print(f"{UNIVERSE_NAME}: {len(picks)} names across {frame['sector'].nunique()} sectors")
frame.head(15)[["symbol", "name", "market_cap", "sector"]]

## 6 · (Optional) Rank by trading activity

For names that already have daily prices in the database, average **dollar
volume** (price × volume) is the cheap "how active is it really?" lens — the
backfill notebook uses it to decide which names deserve slow 1-minute history.

In [ ]:
dv = uv.rank_by_dollar_volume(engine, picks)
if len(dv):
    print(f"{len(dv)} of {len(picks)} picks already have daily data")
    display(dv.head(10).style.format("{:,.0f}"))
else:
    print("No daily data yet for these names — run 02_backfill.ipynb first, "
          "then this ranking becomes meaningful.")

## 7 · Persist: write the YAML + register the symbols

In [ ]:
path = uv.write_universe(UNIVERSE_NAME, picks)
n = uv.register_securities(engine, frame)
print(f"wrote {UNIVERSE_NAME} ({len(picks)} symbols) to {path}")
print(f"registered {n} securities (name/sector/exchange) in the shared database")

## Next steps

* **`02_backfill.ipynb`** — load daily (and optionally 1-minute) history for
  this universe.
* Then point any strategy at it: set `data.universe: LiquidLeaders` in
  `strategies/<name>/config.yaml`.